# Task 12 — Khôi phục nhu cầu tiềm ẩn bằng DLinear (Bảng 2 & xuất Parquet)

**Mục tiêu:** chạy mô hình khôi phục thứ hai là **DLinear** ở hai chế độ để đối chứng với TimesNet (Task 9, Task 10) trên **Bảng 2** của bài báo, và xuất `demand_dlinear.parquet` bàn giao cho **Task 13** (`SSA_DLinear`) và **Task 14** (`DLinear_DLinear`).

| Lượt | Lệnh tác giả tương ứng | Kết quả cần lấy | Số bài báo (Bảng 2) |
|---|---|---|---|
| 1 — MNAR | `python app.py --model DLinear --missing_rate 0.3` | WAPE / WPE trên các ngày có điểm che thêm | WAPE 29,99%, WPE −1,28% |
| 2 — Khôi phục thực tế | `python app.py --model DLinear --missing_rate 0` | `demand_dlinear.parquet` + Decoupling Score | Raw Sales −0,57; DLinear −0,16 |

**Cách làm:** dùng nguyên mã tác giả [frn-50k-baseline @ `8af694a`](https://github.com/Dingdong-Inc/frn-50k-baseline/tree/8af694a09a5a427d1d50648d348c022565a781f2): `CONFIG`, `models.py` (DLinear), `mimic_missing`, `set_seed`, `_imputation` (có chặn âm `np.where(imputation > 0, imputation, 0)`), `evaluation_decoupling`. Kế thừa runner tiết kiệm RAM của Task 9 (đọc `train.parquet` local, X float32, `norm_val` dạng `(n, 1, 6)`). DLinear dùng chung `DatasetForSAITS` và cũng nhân `imputed_data * inputs['norm_val']` như TimesNet, nên mẹo này áp dụng y nguyên. Riêng `_demand_recovery` được viết lại để không tải Hugging Face và không dựng DataFrame chứa mảng 24 giờ.

**Siêu tham số DLinear (đọc trực tiếp từ `models.py`, không sửa):** `n_steps=480`, `n_features=6`, `moving_avg_window_size = patch_len // 2 * 2 + 1 = 17`, **`individual=False`**, `d_model=64`, `ORT_weight=MIT_weight=1`, `epochs=5`, `batch_size=128`, `patience=5`, Adam `lr=0.001`, `weight_decay=1e-5`.
> ⚠️ Bảng phân công ghi `individual=True`, nhưng mã gốc truyền `individual=False` (và `d_model` chỉ có tác dụng ở chế độ không individual). Notebook giữ đúng mã gốc và ghi rõ điều này trong `run_info.md`.

**Trước khi chạy:** *Thời gian chạy → Thay đổi loại thời gian chạy → GPU (T4)*. Chạy lần lượt từ trên xuống. Giữ tab Colab mở trong các lượt đầy đủ.

**Đầu ra trên Drive** (`OUT_DIR`, cấu trúc giống thư mục `Task12_DLinear_Recovery/` trên GitHub):
- `results/dlinear_mnar_metrics.csv`, `results/dlinear_decoupling_metrics.csv`, `results/demand_file_check.csv`
- `results/full/dlinear_mnar.log`, `results/full/dlinear_recovery.log`, `results/smoke/dlinear_smoke.log`, `results/run_metadata.json`, `run_info.md`
- `full/demand_dlinear.parquet` và `checkpoints/` (**chỉ để trên Drive, không đưa lên GitHub**)

## 0. Cấu hình và xác nhận dữ liệu train (SHA-256 Task 6)

In [1]:
from pathlib import Path
import os, sys, json, platform, subprocess, re, time, shutil, hashlib
from datetime import datetime, timezone

from google.colab import drive
drive.mount('/content/drive')

# ---- Sửa nếu file / thư mục của bạn ở chỗ khác ----
DRIVE_TRAIN = Path('/content/drive/MyDrive/PTDLKD/train.parquet')                 # train.parquet chuẩn Task 6
OUT_DIR     = Path('/content/drive/MyDrive/PTDLKD/Task12_DLinear_Recovery')       # nơi lưu kết quả bàn giao

# ---- Tham số chạy (giữ nguyên CONFIG tác giả) ----
MODEL         = 'DLinear'
MNAR_RATE     = 0.3     # Lượt 1: tham số mô phỏng theo ví dụ README tác giả (giống Task 9)
MNAR_SEED     = 1024    # seed CLI gốc -> che đúng 9.163.425 điểm giờ như Task 9
RECOVERY_SEED = 2024    # Lượt 2: theo kế hoạch Task 2, giống Task 10 (đổi 1024 nếu nhóm chốt chạy nguyên bản gốc)
EPOCHS        = 5
BATCH_SIZE    = 128
SMOKE_PAIRS   = 200     # chạy thử 200 cặp đầu (18.000 dòng)

AUTHOR_REPO   = 'https://github.com/Dingdong-Inc/frn-50k-baseline.git'
AUTHOR_COMMIT = '8af694a09a5a427d1d50648d348c022565a781f2'
AUTHOR_DIR    = Path('/content/frn-50k-baseline')
AUTHOR_EXP    = AUTHOR_DIR / 'latent_demand_recovery' / 'exp'
RUNNER        = Path('/content/run_task12.py')
WORK_ROOT     = Path('/content/task12_work')          # file tạm (.npy, parquet smoke) để local, không lên Drive
DATA_PATH     = Path('/content/frn_cache/train.parquet')  # bản sao local của train.parquet (đọc nhanh hơn Drive)
CKPT_MNAR     = OUT_DIR / 'checkpoints' / 'mnar'
CKPT_RECOVERY = OUT_DIR / 'checkpoints' / 'recovery'
DEMAND_LOCAL  = WORK_ROOT / 'full' / 'demand_dlinear.parquet'
DEMAND_DRIVE  = OUT_DIR / 'full' / 'demand_dlinear.parquet'

TASK6_TRAIN_SHA256 = 'b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96'
PAPER_MNAR = {'wape_pct': 29.99, 'wpe_pct': -1.28}            # Bảng 2, DLinear
PAPER_DS   = {'Raw Sales': -0.57, 'DLinear': -0.16}             # Bảng 2, Decoupling Score
TASK9_MASKED_HOURS, TASK9_EVAL_DAYS = 9_163_425, 783_078        # tiêu chí khớp Task 9 (seed 1024)

for sub in ['results/full', 'results/smoke', 'full', 'checkpoints/mnar', 'checkpoints/recovery']:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

assert DRIVE_TRAIN.is_file(), f'Không thấy {DRIVE_TRAIN} – sửa DRIVE_TRAIN cho đúng chỗ lưu train.parquet Task 6.'
shutil.copy2(DRIVE_TRAIN, DATA_PATH)
DATA_SHA256 = sha256_file(DATA_PATH)
print('train.parquet:', DRIVE_TRAIN, f'({DATA_PATH.stat().st_size:,} bytes)')
print('SHA-256      :', DATA_SHA256)
assert DATA_SHA256 == TASK6_TRAIN_SHA256, 'SHA-256 KHÔNG khớp Task 6 – dừng, kiểm tra lại đúng file train.parquet.'
print('Khớp SHA-256 Task 6 (b7e01c71...): True')

Mounted at /content/drive
train.parquet: /content/drive/MyDrive/PTDLKD/train.parquet (61,075,680 bytes)
SHA-256      : b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96
Khớp SHA-256 Task 6 (b7e01c71...): True


## 1. Kiểm tra GPU và RAM

In [2]:
import psutil, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'KHÔNG CÓ GPU')
ram = psutil.virtual_memory().total / 1024**3
print(f'RAM hệ thống: {ram:.1f} GB | torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'Chưa bật GPU: Thời gian chạy → Thay đổi loại thời gian chạy → GPU (T4), rồi chạy lại từ đầu.'

Mon Nov 23 02:11:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Tải mã tác giả và cài thư viện
Giống Task 9: dùng PyTorch/NumPy có sẵn của Colab, chỉ cài thêm các gói mà `pypots` đi kèm repo cần (`--no-deps`). **Không** `pip install pypots` (repo dùng bản `pypots` tùy biến nằm sẵn trong `latent_demand_recovery/`).

In [3]:
if not AUTHOR_DIR.exists():
    subprocess.run(['git', 'clone', '-q', AUTHOR_REPO, str(AUTHOR_DIR)], check=True)
subprocess.run(['git', '-C', str(AUTHOR_DIR), 'checkout', '-q', AUTHOR_COMMIT], check=True)
subprocess.run(['git', '-C', str(AUTHOR_DIR), 'checkout', '--', '.'], check=True)   # đảm bảo mã gốc chưa bị sửa
head = subprocess.run(['git', '-C', str(AUTHOR_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', '-C', str(AUTHOR_DIR), 'status', '--porcelain'], capture_output=True, text=True).stdout.strip()
print('Commit mã tác giả:', head, '| đúng 8af694a:', head == AUTHOR_COMMIT, '| mã gốc không bị sửa:', dirty == '')

def pip(*pkgs):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *pkgs], capture_output=True, text=True)
    print('pip install', *pkgs, '->', 'OK' if r.returncode == 0 else r.stderr[-500:])

pip('pygrinder==0.4', 'tsdb==0.3.1')
for mod, pkg in [('einops', 'einops'), ('h5py', 'h5py'), ('sklearn', 'scikit-learn'), ('sympy', 'sympy'), ('datasets', 'datasets')]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])

Commit mã tác giả: 8af694a09a5a427d1d50648d348c022565a781f2 | đúng 8af694a: True | mã gốc không bị sửa: True
pip install pygrinder==0.4 tsdb==0.3.1 -> OK


In [4]:
# Kiểm tra import pypots đi kèm repo + khởi tạo DLinear đúng như models.py (trong tiến trình riêng)
check = f'''
import sys
sys.path[:0] = [{str(AUTHOR_EXP)!r}, {str(AUTHOR_EXP.parent)!r}]
import pypots, pygrinder, tsdb, torch, numpy, pandas
from model import load_model
import app
cfg = dict(app.CONFIG, model="DLinear", saving_path="/content/tmp_dlinear_check", n_steps=480, n_features=6, patch_len=16, OT=1)
m = load_model(cfg)
print("pypots:", pypots.__file__)
print("DLinear: moving_avg_window_size =", 16 // 2 * 2 + 1, "| individual =", m.individual, "| d_model =", m.d_model,
      "| epochs =", m.epochs, "| batch_size =", m.batch_size, "| patience =", m.original_patience, "| device =", m.device)
print("torch", torch.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__, "| pygrinder", pygrinder.__version__)
'''
r = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-3000:])
if r.returncode != 0 and 'pygrinder' in r.stderr:
    print('>>> pygrinder 0.4 lỗi với môi trường mới; thử bản mới nhất...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-U', 'pygrinder'])
    r = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
    print(r.stdout); print(r.stderr[-3000:])
print('IMPORT OK' if r.returncode == 0 else '>>> IMPORT LỖI – chụp lỗi gửi lại để sửa trước khi chạy tiếp')

cuda:0
pypots: /content/frn-50k-baseline/latent_demand_recovery/pypots/__init__.py
DLinear: moving_avg_window_size = 17 | individual = False | d_model = 64 | epochs = 5 | batch_size = 128 | patience = 5 | device = cuda:0
torch 2.14.1+cu130 | numpy 2.4.6 | pandas 2.2.3 | pygrinder 0.4

I0000 00:00:1795399987.418236    4294 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795399990.318969    4294 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-23 02:13:12 [INFO]: Using the given device: cuda:0
2026-11-23 02:13:12 [INFO]: Model files will be saved to /content/tmp_dlinear_check/DLinear/20261123_T021312
2026-11-23 02:13:12 [INFO]: Tensorboard file will be saved to /content/tmp_dlinear_check/DLinear/20261123_T021312/tenso

## 3. Ghi script chạy
Script nhập `app.py` của tác giả, thay `load_data` / `evaluation_mnar` (giống Task 9) và `_demand_recovery` bằng bản tiết kiệm RAM, in thêm các dòng `[task12]`, rồi gọi đúng `set_seed` → `demand_recovery` như CLI gốc.

`_demand_recovery` viết lại theo đúng lưu ý bắt buộc của bảng phân công: chặn âm (đã có trong `_imputation`), **chỉ thay lát giờ `hours_sale_origin[..., 6:22]` tại đúng các vị trí hết hàng**, giữ nguyên 8 giờ đêm (`[0:6]`, `[22:24]`), cộng đủ 24 giờ thành `sale_amount_pred`; giữ nguyên `hours_sale`, `sale_amount` gốc. Với `--verify_equivalence` (lượt chạy thử), script so với bản gốc của tác giả và dừng nếu khác.

In [5]:
%%writefile /content/run_task12.py
"""Task 12 runner: khôi phục nhu cầu bằng DLinear với mã gốc của tác giả (frn-50k-baseline @ 8af694a).

Kế thừa nguyên runner Task 9 (đã chạy được trên Colab 12,7 GB RAM): mô hình, `mimic_missing`,
`set_seed`, `_imputation` (fit/predict, chặn âm np.where(imputation > 0, imputation, 0)) và
`evaluation_decoupling` dùng nguyên mã tác giả. Viết lại để tiết kiệm RAM, giữ nguyên công thức:
  - load_data, evaluation_mnar: như Task 9 (đọc train.parquet local, X float32, norm_val (n, 1, 6)).
  - _demand_recovery (chỉ chạy khi --missing_rate 0): đọc train.parquet local thay vì Hugging Face,
    chỉ thay lát giờ 6..21 tại đúng các giờ hết hàng bằng giá trị khôi phục, giữ nguyên 8 giờ đêm
    ([0:6], [22:24]) và các giờ quan sát được, cộng đủ 24 giờ thành sale_amount_pred; giữ nguyên
    hours_sale và sale_amount gốc trong Parquet xuất ra (19 cột gốc + sale_amount_pred).
Với --verify_equivalence (chỉ dùng cho tập nhỏ), script so bản viết lại với bản gốc của tác giả
trên cùng dữ liệu, cùng seed và dừng nếu kết quả khác nhau.
"""
import argparse, contextlib, gc, io, json, os, sys, time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

ap = argparse.ArgumentParser()
ap.add_argument('--author_exp', required=True, help='.../frn-50k-baseline/latent_demand_recovery/exp')
ap.add_argument('--data_path', required=True, help='train.parquet')
ap.add_argument('--work_dir', required=True, help='thư mục chạy (checkpoint, imputation .npy)')
ap.add_argument('--model', default='DLinear')
ap.add_argument('--missing_rate', type=float, default=0.3)
ap.add_argument('--seed', type=int, default=1024)
ap.add_argument('--epochs', type=int, default=5)
ap.add_argument('--batch_size', type=int, default=128)
ap.add_argument('--pairs', type=int, default=0, help='0 = toàn bộ; >0 = chỉ lấy N cặp đầu (chạy thử)')
ap.add_argument('--saving_path', default=None, help='thư mục lưu checkpoint (mặc định work_dir/save)')
ap.add_argument('--resume_ckpt', default=None,
                help='file .pypots đã huấn luyện: bỏ qua fit, nạp checkpoint rồi predict + evaluation_mnar')
ap.add_argument('--out_parquet', default=None,
                help='nơi ghi demand parquet khi missing_rate = 0 (mặc định work_dir/demand/demand.parquet)')
ap.add_argument('--verify_equivalence', action='store_true',
                help='so bản gốc và bản viết lại của load_data/evaluation_mnar (chỉ dùng cho tập nhỏ)')
args = ap.parse_args()
args.work_dir = os.path.abspath(args.work_dir)
args.saving_path = os.path.abspath(args.saving_path) if args.saving_path else None
args.resume_ckpt = os.path.abspath(args.resume_ckpt) if args.resume_ckpt else None
args.out_parquet = os.path.abspath(args.out_parquet) if args.out_parquet else None


def ram_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1024 ** 3
    except Exception:
        return float('nan')


SCALAR = ['store_id', 'product_id', 'dt', 'sale_amount', 'discount', 'holiday_flag',
          'precpt', 'avg_temperature', 'stock_hour6_22_cnt']
LISTS = ['hours_sale', 'hours_stock_status']


def read_train(path, n_pairs):
    """Đọc train, sắp xếp theo (store_id, product_id, dt) như mã tác giả."""
    df = pq.read_table(path, columns=SCALAR).to_pandas()
    mats = {}
    for col in LISTS:
        arr = pq.read_table(path, columns=[col]).column(col).combine_chunks()
        lens = pc.list_value_length(arr).to_numpy(zero_copy_only=False)
        if not (lens == 24).all():
            raise ValueError(f'{col}: có dòng không đủ 24 giờ')
        mats[col] = arr.flatten().to_numpy(zero_copy_only=False).reshape(-1, 24)
        del arr
    order = np.lexsort((pd.to_datetime(df['dt']).to_numpy(),
                        df['product_id'].to_numpy(), df['store_id'].to_numpy()))
    df = df.iloc[order].reset_index(drop=True)
    for col in LISTS:
        mats[col] = mats[col][order]
    rows = order                              # vị trí dòng trong file gốc, theo thứ tự đã sắp xếp
    if n_pairs > 0:
        key = df['store_id'].astype(np.int64) * 1_000_000 + df['product_id'].astype(np.int64)
        keep = key.isin(pd.unique(key)[:n_pairs]).to_numpy()
        df = df[keep].reset_index(drop=True)
        for col in LISTS:
            mats[col] = mats[col][keep]
        rows = rows[keep]
    size = df.groupby(['store_id', 'product_id']).size()
    if not (size == 90).all() or len(df) != len(size) * 90:
        raise ValueError('Có cặp không đủ 90 ngày train')
    if not np.isin(mats['hours_stock_status'], [0, 1]).all():
        raise ValueError('hours_stock_status có giá trị khác 0/1')
    mats['hours_stock_status'] = mats['hours_stock_status'].astype(np.int8)   # chỉ so sánh == 1 và cộng
    gc.collect()
    return df, mats, int(len(size)), rows


t_start = time.time()
TRAIN, MATS, N_PAIRS, FILE_ROWS = read_train(args.data_path, args.pairs)
SERIES_NUM = len(TRAIN) // 90
STOCK_POINTS = int(MATS['hours_stock_status'][:, 6:22].sum())
print(f'[task12] data_path={args.data_path}', flush=True)
print(f'[task12] train_rows={len(TRAIN)} pairs={N_PAIRS} '
      f'dt_min={TRAIN["dt"].min()} dt_max={TRAIN["dt"].max()} '
      f'stockout_points_6_21={STOCK_POINTS} ram_gb={ram_gb():.1f}', flush=True)

exp_dir = os.path.abspath(args.author_exp)
sys.path[:0] = [exp_dir, os.path.dirname(exp_dir)]   # exp/ (data, model) và pypots đi kèm repo
os.makedirs(args.work_dir, exist_ok=True)
os.chdir(args.work_dir)                                # app.py ghi ./demand/ và ./save/ vào đây

import data.generate_data as gd      # noqa: E402
import app                           # noqa: E402
import pypots, torch                 # noqa: E402
print(f'[task12] pypots_path={os.path.dirname(pypots.__file__)} torch={torch.__version__} '
      f'cuda={torch.cuda.is_available()} '
      f'gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}', flush=True)

ORIG_LOAD_DATA = app.load_data
ORIG_EVAL_MNAR = app.evaluation_mnar
ORIG_DEMAND_RECOVERY = app._demand_recovery


def load_data_lowmem(CONFIG):
    """Cùng phép tính với generate_data.load_data của tác giả, ít tốn RAM hơn."""
    hours_sale_full = MATS['hours_sale']
    hours_stock_status = MATS['hours_stock_status']
    hours_sale_origin = hours_sale_full.reshape(SERIES_NUM * 3, 30, 24)[..., 6:22]
    hours_stock_status = hours_stock_status.reshape(SERIES_NUM * 3, 30, 24)[..., 6:22]
    hours_sale = np.where(hours_stock_status == 1, np.nan, hours_sale_origin)
    covariate = TRAIN[['discount', 'holiday_flag', 'precpt', 'avg_temperature']].values.reshape(SERIES_NUM * 3, 30, 4)
    covariate = covariate / (covariate.max(axis=1, keepdims=True) + 0.1)

    hours_sale, valid_idx = gd.mimic_missing(hours_sale, p=CONFIG['missing_rate'],
                                             max_missing_patch=7, min_missing_patch=3)

    # Tác giả: concat [doanh số, 4 biến ngoại sinh, giờ/15] ở float64 rồi pypots đổi sang float32.
    # Ở đây cấp phát thẳng mảng float32 và điền từng kênh -> cùng giá trị đầu vào mô hình.
    X = np.empty(hours_sale.shape + (6,), dtype=np.float32)
    X[..., 0] = hours_sale
    X[..., 1:5] = covariate[:, :, None, :]
    X[..., 5] = np.arange(16) / 15
    del hours_sale
    X = X.reshape(-1, 30 * 16, 6)
    valid_idx = valid_idx.reshape(-1, 30 * 16, 1)
    print(valid_idx.sum())
    hours_sale_origin = hours_sale_origin.reshape(-1, 30 * 16, 1)
    # pypots tự tạo norm_val = np.ones(X.shape) ở float64 (3,5 GB) rồi copy sang float32 (1,7 GB) mỗi lần
    # tạo dataset -> hết RAM ngay khi bắt đầu predict. Truyền sẵn norm_val toàn số 1 dạng (n, 1, 6):
    # mô hình nhân dec_out * norm_val theo broadcast nên kết quả y hệt (nhân với 1), chỉ tốn 3,6 MB.
    norm_val = np.ones((X.shape[0], 1, X.shape[-1]), dtype=np.float32)
    out = {
        'train_set': {'X': X, 'norm_val': norm_val},
        'ts_origin': hours_sale_origin,
        'valid_idx': valid_idx,
        'params': {'n_steps': 30 * 16, 'n_features': 6, 'patch_len': 16, 'OT': 1},
    }
    n_total = int(X[..., 0].size)
    n_nan = int(np.isnan(X[..., 0]).sum())
    n_art = int(valid_idx.sum())
    n_stock = n_nan - n_art
    print(f'[task12] n_samples={X.shape[0]} n_points_6_21={n_total} '
          f'stockout_points={n_stock} (khớp dữ liệu: {n_stock == STOCK_POINTS}) '
          f'artificially_masked={n_art} actual_mask_ratio={n_art / (n_total - n_stock):.6f} '
          f'ram_gb={ram_gb():.1f}', flush=True)
    gc.collect()
    return out


def evaluation_mnar_lowmem(X, imputation):
    """Cùng công thức với app.evaluation_mnar của tác giả, dùng mảng đã nạp sẵn."""
    hours_sale_origin = MATS['hours_sale'].reshape(SERIES_NUM * 3, 30, 24)
    stock_hour_X = np.isnan(X[..., 0].reshape(-1, 30, 16)).sum(axis=-1).reshape(-1, 90).reshape(-1)
    stock_hour_origin = MATS['hours_stock_status'][:, 6:22].sum(axis=1)
    hours_sale_impute = hours_sale_origin.copy()
    hours_sale_impute[..., 6:22] = imputation.reshape(-1, 30, 16)
    sale_amount_pred = hours_sale_impute.sum(axis=-1).reshape(-1, 90).reshape(-1)
    valid_idx = (stock_hour_X > 0) & (stock_hour_origin == 0)
    sale_amount = TRAIN['sale_amount'].values
    print('wape', (np.abs(sale_amount_pred - sale_amount) * valid_idx).sum() / (sale_amount * valid_idx).sum())
    print('wpe', ((sale_amount_pred - sale_amount) * valid_idx).sum() / (sale_amount * valid_idx).sum())
    print(f'[task12] eval_days={int(valid_idx.sum())} ram_gb={ram_gb():.1f}', flush=True)


DECOUPLING_COLS = ['city_id', 'store_id', 'product_id', 'dt', 'holiday_flag', 'discount',
                   'sale_amount', 'stock_hour6_22_cnt']


def recovered_daily_demand(imputation):
    """sale_amount_pred: thay giờ 6..21 tại đúng các giờ hết hàng bằng giá trị khôi phục, giữ nguyên
    8 giờ đêm và các giờ quan sát được, rồi cộng đủ 24 giờ (cùng công thức _demand_recovery của tác giả)."""
    if not (imputation >= 0).all():
        raise ValueError('imputation có giá trị âm (lẽ ra _imputation đã chặn âm)')
    hours = MATS['hours_sale'].astype(np.float64).reshape(SERIES_NUM * 3, 30, 24)   # bản sao, MATS giữ nguyên
    stock = MATS['hours_stock_status'].reshape(SERIES_NUM * 3, 30, 24)[..., 6:22] == 1
    business = hours[..., 6:22]                          # view vào lát giờ kinh doanh 6..21
    business[stock] = imputation.reshape(-1, 30, 16)[stock]
    return hours.sum(axis=-1).reshape(-1), int(stock.sum())


def demand_recovery_lowmem(imputation):
    """Thay app._demand_recovery: đọc train.parquet local, ghi Parquet 19 cột gốc + sale_amount_pred."""
    pred, n_replaced = recovered_daily_demand(imputation)
    table = pq.read_table(args.data_path).take(pa.array(FILE_ROWS))    # cùng thứ tự (store_id, product_id, dt)
    for col in ['store_id', 'product_id', 'dt']:
        if not np.array_equal(table.column(col).to_numpy(zero_copy_only=False), TRAIN[col].to_numpy()):
            raise ValueError(f'Thứ tự khóa {col} của Parquet xuất ra không khớp dữ liệu đã nạp')
    table = table.append_column('sale_amount_pred', pa.array(pred, type=pa.float64()))
    out = args.out_parquet or os.path.join(args.work_dir, 'demand', 'demand.parquet')
    os.makedirs(os.path.dirname(out), exist_ok=True)
    pq.write_table(table, out)
    df = table.select(DECOUPLING_COLS + ['sale_amount_pred']).to_pandas()
    n_rows, n_cols = table.num_rows, table.num_columns
    del table
    gc.collect()
    print(f'[task12] demand_written path={out} rows={n_rows} cols={n_cols} '
          f'replaced_stockout_hours={n_replaced} (khớp dữ liệu: {n_replaced == STOCK_POINTS}) '
          f'pred_min={pred.min():.6f} pred_mean={pred.mean():.6f} ram_gb={ram_gb():.1f}', flush=True)
    return df


def verify_recovery_equivalence(CONFIG, X):
    """So sale_amount_pred của bản viết lại với _demand_recovery gốc của tác giả (cùng imputation)."""
    rng = np.random.default_rng(1)
    observed = X[..., :1]
    # Giống đầu ra mô hình: giờ quan sát giữ nguyên X (float32), giờ hết hàng là giá trị khôi phục.
    imp = np.where(np.isnan(observed), rng.random(observed.shape) * 0.5, observed).astype(np.float32)
    pred_new, _ = recovered_daily_demand(imp)
    tmp = os.path.join(args.work_dir, '_verify_recovery')
    os.makedirs(tmp, exist_ok=True)
    cwd = os.getcwd()
    os.chdir(tmp)                                   # bản gốc ghi ./demand/demand.parquet -> để riêng chỗ khác
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            ref = ORIG_DEMAND_RECOVERY(imp)
    finally:
        os.chdir(cwd)
    same_keys = all(np.array_equal(ref[c].to_numpy(), TRAIN[c].to_numpy()) for c in ['store_id', 'product_id', 'dt'])
    diff = float(np.abs(ref['sale_amount_pred'].to_numpy() - pred_new).max())
    # Chênh lệch duy nhất được phép: tác giả ghi đè cả lát 6..21 bằng X float32 ở giờ quan sát,
    # bản viết lại giữ nguyên giá trị float64 gốc -> chỉ lệch do làm tròn float32.
    ok = same_keys and diff < 1e-4
    print(f'[task12] recovery_equivalence keys={same_keys} max_abs_diff_sale_amount_pred={diff:.3e} '
          f'recovery_equivalence_ok={ok}', flush=True)
    if not ok:
        raise SystemExit('Bản viết lại _demand_recovery KHÔNG trùng bản gốc – dừng, không huấn luyện.')


def verify_equivalence(CONFIG):
    """So bản gốc của tác giả với bản viết lại trên cùng dữ liệu, cùng seed."""
    df_hf = TRAIN.copy()                     # giả lập dataset['train'].to_pandas() của Hugging Face
    for col in LISTS:
        obj = np.empty(len(df_hf), dtype=object)
        m = MATS[col].astype(np.int64) if col == 'hours_stock_status' else MATS[col]
        for i in range(len(df_hf)):
            obj[i] = m[i]
        df_hf[col] = obj

    class _Split:
        def to_pandas(self):
            return df_hf.sample(frac=1, random_state=0)   # xáo trộn: tác giả tự sort lại

    fake_hf = lambda *a, **k: {'train': _Split()}
    gd.load_dataset = fake_hf
    app.load_dataset = fake_hf
    quiet = io.StringIO
    with contextlib.redirect_stdout(quiet()):
        app.set_seed(seed_value=args.seed)
        ref = ORIG_LOAD_DATA(CONFIG)
        app.set_seed(seed_value=args.seed)
        new = load_data_lowmem(CONFIG)
    same_X = np.array_equal(ref['train_set']['X'].astype(np.float32), new['train_set']['X'], equal_nan=True)
    same_mask = np.array_equal(ref['valid_idx'], new['valid_idx'])
    same_origin = np.array_equal(ref['ts_origin'], new['ts_origin'])
    rng = np.random.default_rng(0)
    imp = (rng.random(new['train_set']['X'].shape[:2] + (1,)) * 0.3).astype(np.float32)
    buf_ref, buf_new = quiet(), quiet()
    with contextlib.redirect_stdout(buf_ref):
        ORIG_EVAL_MNAR(ref['train_set']['X'], imp)
    with contextlib.redirect_stdout(buf_new):
        evaluation_mnar_lowmem(new['train_set']['X'], imp)
    lines_ref = [l for l in buf_ref.getvalue().splitlines() if l.startswith('wape') or l.startswith('wpe')]
    lines_new = [l for l in buf_new.getvalue().splitlines() if l.startswith('wape') or l.startswith('wpe')]
    same_eval = lines_ref == lines_new and len(lines_ref) == 2
    ok = same_X and same_mask and same_origin and same_eval
    print(f'[task12] equivalence X={same_X} valid_idx={same_mask} ts_origin={same_origin} '
          f'evaluation_mnar={same_eval} ({lines_ref} vs {lines_new}) equivalence_ok={ok}', flush=True)
    if ok and CONFIG['missing_rate'] == 0:
        verify_recovery_equivalence(CONFIG, new['train_set']['X'])
    gd.load_dataset = app.load_dataset = None
    del df_hf, ref, new
    gc.collect()
    if not ok:
        raise SystemExit('Bản viết lại KHÔNG trùng bản gốc – dừng, không huấn luyện.')


app.load_data = load_data_lowmem
app.evaluation_mnar = evaluation_mnar_lowmem
app._demand_recovery = demand_recovery_lowmem

app.CONFIG.update({
    'model': args.model,
    'missing_rate': args.missing_rate,
    'EPOCHS': args.epochs,
    'batch_size': args.batch_size,
    'saving_path': args.saving_path or os.path.join(args.work_dir, 'save'),
})
print('[task12] CONFIG ' + json.dumps(app.CONFIG, ensure_ascii=False), flush=True)
print(f'[task12] seed={args.seed}', flush=True)

if args.verify_equivalence:
    verify_equivalence(app.CONFIG)

_orig_load_model = app.load_model


def load_model_logged(CONFIG):
    m = _orig_load_model(CONFIG)
    _orig_fit, _orig_predict = m.fit, m.predict

    def fit_logged(*a, **k):
        print(f'[task12] fit_start ram_gb={ram_gb():.1f}', flush=True)
        if args.resume_ckpt:
            # Tiếp tục sau khi bị ngắt: giữ nguyên _imputation của tác giả, thay huấn luyện bằng nạp checkpoint.
            # set_seed trước load_data như CLI gốc -> cùng mặt nạ thiếu như lượt đã huấn luyện.
            loaded = torch.load(args.resume_ckpt, map_location=m.device, weights_only=False)
            state = loaded.state_dict() if isinstance(loaded, torch.nn.Module) else loaded.model.state_dict()
            m.model.load_state_dict(state)
            print(f'[task12] resumed_from_checkpoint={args.resume_ckpt} (bỏ qua huấn luyện)', flush=True)
        else:
            _orig_fit(*a, **k)
        print(f'[task12] fit_end ram_gb={ram_gb():.1f}', flush=True)

    def predict_logged(*a, **k):
        print(f'[task12] predict_start ram_gb={ram_gb():.1f}', flush=True)
        res = _orig_predict(*a, **k)
        print(f'[task12] predict_end shape={res["imputation"].shape} ram_gb={ram_gb():.1f}', flush=True)
        return res

    m.fit, m.predict = fit_logged, predict_logged
    return m


app.load_model = load_model_logged

app.set_seed(seed_value=args.seed)   # như CLI gốc: set_seed rồi demand_recovery
app.demand_recovery(app.CONFIG)      # >0: _imputation -> evaluation_mnar (wape, wpe); =0: _demand_recovery -> evaluation_decoupling
print(f'[task12] finished elapsed_sec={time.time() - t_start:.0f} ram_gb={ram_gb():.1f}', flush=True)

Writing /content/run_task12.py


In [6]:
def run_logged(extra_args, log_path, work_dir, missing_rate, seed, append=False):
    cmd = [sys.executable, '-u', str(RUNNER), '--author_exp', str(AUTHOR_EXP), '--data_path', str(DATA_PATH),
           '--work_dir', str(work_dir), '--model', MODEL, '--missing_rate', str(missing_rate),
           '--seed', str(seed), '--batch_size', str(BATCH_SIZE), *extra_args]
    print('Lệnh:', ' '.join(cmd), '\n')
    t0 = time.time()
    with open(log_path, 'a' if append else 'w', encoding='utf-8') as log:
        log.write('# command: ' + ' '.join(cmd) + '\n')
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='')
            log.write(line); log.flush()
        p.wait()
        log.write(f'# exit_code: {p.returncode}\n')
    print(f'\n== mã thoát {p.returncode}, {time.time() - t0:.0f} giây. Log: {log_path}')
    return cmd, p.returncode

NUM = r'(-?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?|nan)'
def parse_log(text):
    def grab(pattern, cast=float):
        m = re.findall(pattern, text, flags=re.M)
        return cast(m[-1]) if m else None
    flag = lambda pat: (lambda v: None if v is None else v == 'True')(grab(pat + r'=(True|False)', str))
    return {
        'wape': grab(r'^wape\s+' + NUM), 'wpe': grab(r'^wpe\s+' + NUM),
        'decoupling_raw': grab(r'\bsale_amount\s+' + NUM), 'decoupling_pred': grab(r'\bsale_amount_pred\s+' + NUM),
        'artificially_masked': grab(r'artificially_masked=(\d+)', int),
        'actual_mask_ratio': grab(r'actual_mask_ratio=' + NUM),
        'stockout_points': grab(r'stockout_points=(\d+)', int),
        'n_samples': grab(r'n_samples=(\d+)', int), 'train_rows': grab(r'train_rows=(\d+)', int),
        'eval_days': grab(r'eval_days=(\d+)', int),
        'demand_rows': grab(r'demand_written .*?rows=(\d+)', int), 'demand_cols': grab(r'demand_written .*?cols=(\d+)', int),
        'replaced_stockout_hours': grab(r'replaced_stockout_hours=(\d+)', int),
        'equivalence_ok': flag(r'(?<!_)equivalence_ok'), 'recovery_equivalence_ok': flag('recovery_equivalence_ok'),
        'recovery_max_abs_diff': grab(r'max_abs_diff_sale_amount_pred=' + NUM),
        'elapsed_sec': grab(r'finished elapsed_sec=(\d+)', int),
        'peak_ram_gb': max([float(x) for x in re.findall(r'ram_gb=([\d.]+)', text)] or [float('nan')]),
    }

def find_ckpt(ckpt_dir):
    found = [p for p in Path(ckpt_dir).rglob(f'{MODEL}.pypots') if 'tensorboard' not in p.parts]
    return max(found, key=lambda p: p.stat().st_mtime) if found else None

## 4. Chạy thử (smoke) trên 200 cặp — cả hai chế độ
Mục đích: kiểm tra môi trường và luồng chạy, **không** dùng số này làm kết quả. Hai lượt ghi chung vào `results/smoke/dlinear_smoke.log`. Lượt 2 ghi Parquet thử ra **`/content/task12_work/smoke/demand_smoke_200pairs.parquet`** (chỉ để local, **tuyệt đối không bàn giao file này**).

In [7]:
SMOKE_LOG = OUT_DIR / 'results/smoke/dlinear_smoke.log'
smoke_mnar_cmd, smoke_mnar_rc = run_logged(
    ['--pairs', str(SMOKE_PAIRS), '--epochs', str(EPOCHS), '--verify_equivalence'],
    SMOKE_LOG, WORK_ROOT / 'smoke_mnar', MNAR_RATE, MNAR_SEED)
smoke_mnar = parse_log(SMOKE_LOG.read_text(encoding='utf-8'))
mark = len(SMOKE_LOG.read_text(encoding='utf-8'))

smoke_rec_cmd, smoke_rec_rc = run_logged(
    ['--pairs', str(SMOKE_PAIRS), '--epochs', str(EPOCHS), '--verify_equivalence',
     '--out_parquet', str(WORK_ROOT / 'smoke' / 'demand_smoke_200pairs.parquet')],
    SMOKE_LOG, WORK_ROOT / 'smoke_recovery', 0, RECOVERY_SEED, append=True)
smoke_rec = parse_log(SMOKE_LOG.read_text(encoding='utf-8')[mark:])

print(json.dumps({'mnar': smoke_mnar, 'recovery': smoke_rec}, indent=2))
ok = (smoke_mnar_rc == 0 and smoke_mnar['wape'] is not None and smoke_mnar['equivalence_ok'] is True
      and smoke_rec_rc == 0 and smoke_rec['equivalence_ok'] is True and smoke_rec['recovery_equivalence_ok'] is True
      and smoke_rec['demand_rows'] == SMOKE_PAIRS * 90 and smoke_rec['demand_cols'] == 20
      and smoke_rec['decoupling_pred'] is not None)
print('\nCHẠY THỬ OK (equivalence_ok=True ở cả hai chế độ) – chạy tiếp mục 5' if ok
      else '\n>>> CHẠY THỬ LỖI – chụp phần cuối log gửi lại, chưa chạy mục 5')

Lệnh: /usr/bin/python3 -u /content/run_task12.py --author_exp /content/frn-50k-baseline/latent_demand_recovery/exp --data_path /content/frn_cache/train.parquet --work_dir /content/task12_work/smoke_mnar --model DLinear --missing_rate 0.3 --seed 1024 --batch_size 128 --pairs 200 --epochs 5 --verify_equivalence 

[task12] data_path=/content/frn_cache/train.parquet
[task12] train_rows=18000 pairs=200 dt_min=2024-03-28 dt_max=2024-06-25 stockout_points_6_21=57796 ram_gb=0.4
I0000 00:00:1795400168.113514    5030 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795400170.945327    5030 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
cuda:0
[task12] pypots_path=/content/frn-50k-baseline/latent_demand_recovery/pypots torch=2.14.1+c

## 5. Lượt 1 — MNAR đầy đủ (`--missing_rate 0.3`, seed 1024, 50.000 cặp = 150.000 mẫu)
Theo dõi các dòng `Epoch 00x` và `[task12] ... ram_gb=`. Checkpoint lưu thẳng lên Drive (`checkpoints/mnar/`).

In [8]:
MNAR_LOG = OUT_DIR / 'results/full/dlinear_mnar.log'
mnar_cmd, mnar_rc = run_logged(['--epochs', str(EPOCHS), '--saving_path', str(CKPT_MNAR)],
                               MNAR_LOG, WORK_ROOT / 'full_mnar', MNAR_RATE, MNAR_SEED)
mnar = parse_log(MNAR_LOG.read_text(encoding='utf-8'))
MNAR_RESUMED = None
print(json.dumps(mnar, indent=2))
print('Checkpoint:', find_ckpt(CKPT_MNAR) or 'chưa có')

Lệnh: /usr/bin/python3 -u /content/run_task12.py --author_exp /content/frn-50k-baseline/latent_demand_recovery/exp --data_path /content/frn_cache/train.parquet --work_dir /content/task12_work/full_mnar --model DLinear --missing_rate 0.3 --seed 1024 --batch_size 128 --epochs 5 --saving_path /content/drive/MyDrive/PTDLKD/Task12_DLinear_Recovery/checkpoints/mnar 

[task12] data_path=/content/frn_cache/train.parquet
[task12] train_rows=4500000 pairs=50000 dt_min=2024-03-28 dt_max=2024-06-25 stockout_points_6_21=14311536 ram_gb=1.6
I0000 00:00:1795400308.522730    5654 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795400310.894633    5654 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
cuda:0
[task12] pypots_path=/content/frn

### 5b. CHỈ chạy nếu mục 5 bị ngắt SAU dòng `Saved the model` (chưa có `wape`/`wpe`)
Chạy lại mục 0 → 3 và ô định nghĩa hàm (mục 3), rồi chạy ô này: nạp checkpoint trên Drive, dự đoán và tính WAPE/WPE mà không huấn luyện lại. Phải dùng đúng tài khoản Google đã chạy mục 5.

In [ ]:
MNAR_RESUMED = find_ckpt(CKPT_MNAR)
assert MNAR_RESUMED, 'Không thấy DLinear.pypots trong checkpoints/mnar – phải chạy lại mục 5.'
MNAR_LOG = OUT_DIR / 'results/full/dlinear_mnar_predict_from_ckpt.log'
mnar_cmd, mnar_rc = run_logged(['--epochs', str(EPOCHS), '--resume_ckpt', str(MNAR_RESUMED)],
                               MNAR_LOG, WORK_ROOT / 'full_mnar', MNAR_RATE, MNAR_SEED)
mnar = parse_log(MNAR_LOG.read_text(encoding='utf-8'))
print(json.dumps(mnar, indent=2))

## 6. Lượt 2 — Khôi phục thực tế đầy đủ (`--missing_rate 0`)
Huấn luyện trên toàn bộ train, khôi phục nhu cầu tại các giờ hết hàng thực tế, ghi `demand_dlinear.parquet` (ghi local trước rồi sao chép lên Drive), và in Decoupling Score bằng `evaluation_decoupling` gốc của tác giả.

In [9]:
REC_LOG = OUT_DIR / 'results/full/dlinear_recovery.log'
rec_cmd, rec_rc = run_logged(['--epochs', str(EPOCHS), '--saving_path', str(CKPT_RECOVERY), '--out_parquet', str(DEMAND_LOCAL)],
                             REC_LOG, WORK_ROOT / 'full_recovery', 0, RECOVERY_SEED)
rec = parse_log(REC_LOG.read_text(encoding='utf-8'))
REC_RESUMED = None
print(json.dumps(rec, indent=2))
assert rec_rc == 0 and rec['demand_rows'] == 4_500_000, 'Lượt khôi phục chưa xong – xem log (nếu đã có "Saved the model", chạy ô 6b).'
shutil.copy2(DEMAND_LOCAL, DEMAND_DRIVE)
print('Đã sao chép lên Drive:', DEMAND_DRIVE, f'({DEMAND_DRIVE.stat().st_size:,} bytes)')

Lệnh: /usr/bin/python3 -u /content/run_task12.py --author_exp /content/frn-50k-baseline/latent_demand_recovery/exp --data_path /content/frn_cache/train.parquet --work_dir /content/task12_work/full_recovery --model DLinear --missing_rate 0 --seed 2024 --batch_size 128 --epochs 5 --saving_path /content/drive/MyDrive/PTDLKD/Task12_DLinear_Recovery/checkpoints/recovery --out_parquet /content/task12_work/full/demand_dlinear.parquet 

[task12] data_path=/content/frn_cache/train.parquet
[task12] train_rows=4500000 pairs=50000 dt_min=2024-03-28 dt_max=2024-06-25 stockout_points_6_21=14311536 ram_gb=1.6
I0000 00:00:1795400834.098249    7827 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795400836.457317    7827 cudart_stub.cc:31] Could not find cuda drivers on your ma

### 6b. CHỈ chạy nếu mục 6 bị ngắt SAU dòng `Saved the model` (chưa có file Parquet / Decoupling Score)

In [ ]:
REC_RESUMED = find_ckpt(CKPT_RECOVERY)
assert REC_RESUMED, 'Không thấy DLinear.pypots trong checkpoints/recovery – phải chạy lại mục 6.'
REC_LOG = OUT_DIR / 'results/full/dlinear_recovery_predict_from_ckpt.log'
rec_cmd, rec_rc = run_logged(['--epochs', str(EPOCHS), '--resume_ckpt', str(REC_RESUMED), '--out_parquet', str(DEMAND_LOCAL)],
                             REC_LOG, WORK_ROOT / 'full_recovery', 0, RECOVERY_SEED)
rec = parse_log(REC_LOG.read_text(encoding='utf-8'))
print(json.dumps(rec, indent=2))
assert rec_rc == 0 and rec['demand_rows'] == 4_500_000
shutil.copy2(DEMAND_LOCAL, DEMAND_DRIVE)
print('Đã sao chép lên Drive:', DEMAND_DRIVE, f'({DEMAND_DRIVE.stat().st_size:,} bytes)')

## 7. Bảng chỉ số MNAR (`dlinear_mnar_metrics.csv`) — đối chiếu Bảng 2
Tiêu chí: với seed 1024 phải che đúng **9.163.425** điểm giờ và đánh giá trên **783.078** ngày như Task 9.

In [10]:
import math, csv
import pandas as pd
finite = lambda x: x is not None and isinstance(x, float) and math.isfinite(x)

mnar_status = 'ok' if (mnar_rc == 0 and finite(mnar['wape']) and finite(mnar['wpe']) and mnar['train_rows'] == 4_500_000) else 'failed'
row = {
    'model': MODEL, 'missing_rate': MNAR_RATE, 'seed': MNAR_SEED, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
    'n_artificially_masked_hours': mnar['artificially_masked'] if mnar['artificially_masked'] is not None else '',
    'wape_pct': round(mnar['wape'] * 100, 4) if mnar_status == 'ok' else '',
    'wpe_pct': round(mnar['wpe'] * 100, 4) if mnar_status == 'ok' else '',
    'status': mnar_status,
    'log_path': str(MNAR_LOG.relative_to(OUT_DIR)),
}
with open(OUT_DIR / 'results/dlinear_mnar_metrics.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerow(row)
display(pd.read_csv(OUT_DIR / 'results/dlinear_mnar_metrics.csv'))

print(f"Số điểm giờ che thêm: {mnar['artificially_masked'] or 0:,} | khớp Task 9 ({TASK9_MASKED_HOURS:,}): {mnar['artificially_masked'] == TASK9_MASKED_HOURS}")
print(f"Số ngày đánh giá   : {mnar['eval_days'] or 0:,} | khớp Task 9 ({TASK9_EVAL_DAYS:,}): {mnar['eval_days'] == TASK9_EVAL_DAYS}")
if mnar_status == 'ok':
    print(f"Bảng 2 (DLinear): WAPE nhóm {row['wape_pct']:.2f}% vs bài báo {PAPER_MNAR['wape_pct']:.2f}% "
          f"(chênh {row['wape_pct'] - PAPER_MNAR['wape_pct']:+.2f} pp); WPE nhóm {row['wpe_pct']:.2f}% vs "
          f"{PAPER_MNAR['wpe_pct']:.2f}% (chênh {row['wpe_pct'] - PAPER_MNAR['wpe_pct']:+.2f} pp)")
assert mnar['artificially_masked'] == TASK9_MASKED_HOURS and mnar['eval_days'] == TASK9_EVAL_DAYS, \
    'Số điểm che / số ngày đánh giá không khớp Task 9 – kiểm tra lại seed và file train.'

,model,missing_rate,seed,epochs,batch_size,n_artificially_masked_hours,wape_pct,wpe_pct,status,log_path
0,DLinear,0.3,1024,5,128,9163425,30.3212,-14.3819,ok,results/full/dlinear_mnar.log


Số điểm giờ che thêm: 9,163,425 | khớp Task 9 (9,163,425): True
Số ngày đánh giá   : 783,078 | khớp Task 9 (783,078): True
Bảng 2 (DLinear): WAPE nhóm 30.32% vs bài báo 29.99% (chênh +0.33 pp); WPE nhóm -14.38% vs -1.28% (chênh -13.10 pp)


## 8. Đối soát toàn vẹn `demand_dlinear.parquet` (`demand_file_check.csv`)
Cùng tiêu chí với Cell 7 của Task 10, nhưng đọc bằng PyArrow theo cột để không tràn RAM T4: 4.500.000 dòng × 20 cột, 0 khóa trùng / thiếu / thừa, `sale_amount_pred` không thiếu, không vô cực, không âm; `sale_amount` và **toàn bộ** `hours_sale` gốc giữ nguyên.

In [11]:
import numpy as np, pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
KEY = ['store_id', 'product_id', 'dt']
TARGET = 'sale_amount_pred'

train_cols = pq.ParquetFile(DATA_PATH).schema_arrow.names
demand_cols = pq.ParquetFile(DEMAND_DRIVE).schema_arrow.names
assert len(train_cols) == 19 and demand_cols == train_cols + [TARGET], f'Schema không khớp: {demand_cols}'

tr = pq.read_table(DATA_PATH, columns=KEY + ['sale_amount']).to_pandas()
dm = pq.read_table(DEMAND_DRIVE, columns=KEY + ['sale_amount', TARGET]).to_pandas()
order = np.lexsort((pd.to_datetime(tr['dt']).to_numpy(), tr['product_id'].to_numpy(), tr['store_id'].to_numpy()))
tr_sorted = tr.iloc[order].reset_index(drop=True)

n_dup = int(dm.duplicated(subset=KEY).sum())
tr_idx, dm_idx = pd.MultiIndex.from_frame(tr[KEY]), pd.MultiIndex.from_frame(dm[KEY])
n_missing, n_extra = int(len(tr_idx.difference(dm_idx))), int(len(dm_idx.difference(tr_idx)))
pred = pd.to_numeric(dm[TARGET], errors='coerce')
n_pred_missing = int(pred.isna().sum())
n_pred_non_finite = int((~np.isfinite(pred.dropna())).sum())
n_pred_negative = int((pred.dropna() < 0).sum())

same_order = tr_sorted[KEY].equals(dm[KEY])
max_sale_diff = float((tr_sorted['sale_amount'] - dm['sale_amount']).abs().max())
hs_tr = pq.read_table(DATA_PATH, columns=['hours_sale']).column('hours_sale').combine_chunks()
hs_tr = hs_tr.flatten().to_numpy(zero_copy_only=False).reshape(-1, 24)[order]
hs_dm = pq.read_table(DEMAND_DRIVE, columns=['hours_sale']).column('hours_sale').combine_chunks()
hs_dm = hs_dm.flatten().to_numpy(zero_copy_only=False).reshape(-1, 24)
hours_equal = bool(np.array_equal(hs_tr, hs_dm))
st = pq.read_table(DATA_PATH, columns=['stock_hour6_22_cnt']).column(0).to_numpy()[order]
nostock = st == 0
# Ngày không hết hàng: nhu cầu khôi phục = tổng 24 giờ quan sát được (không bị mô hình thay đổi)
nostock_equal = bool(np.allclose(pred.to_numpy()[nostock], hs_tr[nostock].sum(axis=1), rtol=0, atol=1e-9))
del hs_tr, hs_dm

DEMAND_SHA256 = sha256_file(DEMAND_DRIVE)
DEMAND_BYTES = DEMAND_DRIVE.stat().st_size
check = pd.DataFrame([{
    'file_path': str(DEMAND_DRIVE), 'n_rows': len(dm), 'n_cols': len(demand_cols),
    'n_duplicate_keys': n_dup, 'n_missing_keys_vs_train': n_missing, 'n_extra_keys_vs_train': n_extra,
    'n_pred_missing': n_pred_missing, 'n_pred_non_finite': n_pred_non_finite, 'n_pred_negative': n_pred_negative,
}])
check.to_csv(OUT_DIR / 'results/demand_file_check.csv', index=False)
display(check)
print(f'SHA-256: {DEMAND_SHA256} | {DEMAND_BYTES:,} bytes')
print(f'Thứ tự khóa khớp train đã sort: {same_order} | max |sale_amount gốc - demand|: {max_sale_diff}')
print(f'hours_sale giữ nguyên (toàn bộ 4.500.000 dòng): {hours_equal} | ngày không hết hàng giữ nguyên tổng quan sát: {nostock_equal}')
print(dm[['sale_amount', TARGET]].describe())
err = ['n_duplicate_keys', 'n_missing_keys_vs_train', 'n_extra_keys_vs_train', 'n_pred_missing', 'n_pred_non_finite', 'n_pred_negative']
assert len(dm) == 4_500_000 and (check[err].iloc[0] == 0).all(), 'PHÁT HIỆN LỖI ĐỐI SOÁT – không bàn giao file này.'
assert same_order and max_sale_diff == 0.0 and hours_equal and nostock_equal, 'Cột gốc bị thay đổi – không bàn giao.'
print('PASSED: demand_dlinear.parquet đạt mọi tiêu chí (toàn bộ bộ đếm lỗi = 0).')

,file_path,n_rows,n_cols,n_duplicate_keys,n_missing_keys_vs_train,n_extra_keys_vs_train,n_pred_missing,n_pred_non_finite,n_pred_negative
0,/content/drive/MyDrive/PTDLKD/Task12_DLinear_R...,4500000,20,0,0,0,0,0,0


SHA-256: 2cd9091adec89b129d0c726e3be658b9becf027b4ece1e3b725feb74176ab65b | 81,531,473 bytes
Thứ tự khóa khớp train đã sort: True | max |sale_amount gốc - demand|: 0.0
hours_sale giữ nguyên (toàn bộ 4.500.000 dòng): True | ngày không hết hàng giữ nguyên tổng quan sát: True
        sale_amount  sale_amount_pred
count  4.500000e+06      4.500000e+06
mean   9.985913e-01      1.143788e+00
std    1.406738e+00      1.516064e+00
min    0.000000e+00      0.000000e+00
25%    4.000000e-01      5.000000e-01
50%    7.000000e-01      8.000000e-01
75%    1.100000e+00      1.300000e+00
max    4.490000e+01      4.490000e+01
PASSED: demand_dlinear.parquet đạt mọi tiêu chí (toàn bộ bộ đếm lỗi = 0).


## 9. Decoupling Score (`dlinear_decoupling_metrics.csv`) — đối chiếu Bảng 2

In [12]:
ds_rows = [
    {'method': 'Raw Sales', 'decoupling_score': rec['decoupling_raw'], 'log_path': str(REC_LOG.relative_to(OUT_DIR))},
    {'method': 'DLinear',   'decoupling_score': rec['decoupling_pred'], 'log_path': str(REC_LOG.relative_to(OUT_DIR))},
]
assert all(finite(r['decoupling_score']) for r in ds_rows), 'Không tìm thấy Decoupling Score trong log.'
ds = pd.DataFrame(ds_rows)
ds.to_csv(OUT_DIR / 'results/dlinear_decoupling_metrics.csv', index=False)
display(ds)
for r in ds_rows:
    print(f"{r['method']:9s}: nhóm {r['decoupling_score']:+.4f} ({r['decoupling_score']:+.2f}) | bài báo {PAPER_DS[r['method']]:+.2f} "
          f"| chênh {round(r['decoupling_score'], 2) - PAPER_DS[r['method']]:+.2f}")
print('\nRaw Sales phải trùng Task 10 (−0,566447) vì chỉ phụ thuộc dữ liệu gốc.')

,method,decoupling_score,log_path
0,Raw Sales,-0.566447,results/full/dlinear_recovery.log
1,DLinear,-0.151151,results/full/dlinear_recovery.log


Raw Sales: nhóm -0.5664 (-0.57) | bài báo -0.57 | chênh +0.00
DLinear  : nhóm -0.1512 (-0.15) | bài báo -0.16 | chênh +0.01

Raw Sales phải trùng Task 10 (−0,566447) vì chỉ phụ thuộc dữ liệu gốc.


## 10. Chia sẻ file lớn trên Drive
1. Mở Google Drive → `PTDLKD/Task12_DLinear_Recovery/full/demand_dlinear.parquet` → **Chia sẻ** → *Quyền truy cập chung* → **Bất kỳ ai có đường liên kết** (Người xem) → Sao chép đường liên kết. Làm tương tự với thư mục `checkpoints/`.
2. Dán hai link vào ô dưới, chạy ô. **Mở thử link `demand_dlinear.parquet` bằng cửa sổ ẩn danh** để chắc tải được.
3. Gửi link ngay cho **Phạm Đăng Khoa (Task 13)** và **Dương Trọng Toàn (Task 14)**, kèm SHA-256 in ở mục 8.

In [16]:
DEMAND_DRIVE_LINK = 'https://drive.google.com/file/d/1MDcD7cOyzcnoBu48I6eGUkSfwx8b3hkh/view?usp=drive_link'
CKPT_DRIVE_LINK   = 'https://drive.google.com/drive/folders/18MjIAn0RF705AxhluhvgomYmNepY5kTo?usp=sharing'
assert DEMAND_DRIVE_LINK.startswith('https://drive.google.com/'), 'Chưa dán link chia sẻ demand_dlinear.parquet.'
print('Link demand:', DEMAND_DRIVE_LINK); print('Link checkpoint:', CKPT_DRIVE_LINK)

Link demand: https://drive.google.com/file/d/1MDcD7cOyzcnoBu48I6eGUkSfwx8b3hkh/view?usp=drive_link
Link checkpoint: https://drive.google.com/drive/folders/18MjIAn0RF705AxhluhvgomYmNepY5kTo?usp=sharing


## 11. Lưu metadata và `run_info.md`

In [17]:
import importlib.metadata as im
def ver(p):
    try: return im.version(p)
    except Exception: return 'n/a'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
ENV = {'python': platform.python_version(), 'torch': ver('torch'), 'numpy': ver('numpy'), 'pandas': ver('pandas'),
       'pyarrow': ver('pyarrow'), 'pygrinder': ver('pygrinder'), 'tsdb': ver('tsdb'), 'gpu': gpu,
       'ram_gb': round(psutil.virtual_memory().total / 1024**3, 1)}
DLINEAR_PARAMS = {'n_steps': 480, 'n_features': 6, 'moving_avg_window_size': 17, 'individual': False, 'd_model': 64,
                  'ORT_weight': 1, 'MIT_weight': 1, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'patience': 5,
                  'optimizer': 'Adam', 'lr': 0.001, 'weight_decay': 1e-5}
RUN_META = {
    'task': 'Task 12 - DLinear latent demand recovery', 'run_utc': datetime.now(timezone.utc).isoformat(),
    'author_repo': AUTHOR_REPO, 'author_commit': head,
    'data': {'train_parquet': str(DRIVE_TRAIN), 'sha256': DATA_SHA256, 'same_as_task6': DATA_SHA256 == TASK6_TRAIN_SHA256},
    'dlinear_params': DLINEAR_PARAMS, 'environment': ENV,
    'smoke': {'pairs': SMOKE_PAIRS, 'mnar': {'returncode': smoke_mnar_rc, **smoke_mnar}, 'recovery': {'returncode': smoke_rec_rc, **smoke_rec}},
    'mnar': {'missing_rate': MNAR_RATE, 'seed': MNAR_SEED, 'returncode': mnar_rc, **mnar, 'status': mnar_status,
             'resumed_from_checkpoint': str(MNAR_RESUMED) if MNAR_RESUMED else None, 'command': ' '.join(mnar_cmd)},
    'recovery': {'missing_rate': 0.0, 'seed': RECOVERY_SEED, 'returncode': rec_rc, **rec,
                 'resumed_from_checkpoint': str(REC_RESUMED) if REC_RESUMED else None, 'command': ' '.join(rec_cmd),
                 'demand_parquet': {'drive_path': str(DEMAND_DRIVE), 'bytes': DEMAND_BYTES, 'sha256': DEMAND_SHA256,
                                    'share_link': DEMAND_DRIVE_LINK, 'checkpoint_link': CKPT_DRIVE_LINK}},
}
(OUT_DIR / 'results/run_metadata.json').write_text(json.dumps(RUN_META, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(RUN_META, indent=2, ensure_ascii=False)[:3000])

{
  "task": "Task 12 - DLinear latent demand recovery",
  "run_utc": "2026-11-23T02:48:40.468540+00:00",
  "author_repo": "https://github.com/Dingdong-Inc/frn-50k-baseline.git",
  "author_commit": "8af694a09a5a427d1d50648d348c022565a781f2",
  "data": {
    "train_parquet": "/content/drive/MyDrive/PTDLKD/train.parquet",
    "sha256": "b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96",
    "same_as_task6": true
  },
  "dlinear_params": {
    "n_steps": 480,
    "n_features": 6,
    "moving_avg_window_size": 17,
    "individual": false,
    "d_model": 64,
    "ORT_weight": 1,
    "MIT_weight": 1,
    "epochs": 5,
    "batch_size": 128,
    "patience": 5,
    "optimizer": "Adam",
    "lr": 0.001,
    "weight_decay": 1e-05
  },
  "environment": {
    "python": "3.13.16",
    "torch": "2.14.1+cu130",
    "numpy": "2.4.6",
    "pandas": "2.2.3",
    "pyarrow": "23.0.1",
    "pygrinder": "0.4",
    "tsdb": "0.3.1",
    "gpu": "Tesla T4, 15360 MiB",
    "ram_gb": 12.7
  },
  "sm

In [18]:
pct = lambda x: f'{x * 100:.2f}%' if finite(x) else 'không có'
F = '`' * 3
info = f'''# Thông tin chạy Task 12 — DLinear: khôi phục nhu cầu tiềm ẩn

**Cách tái lập:** mở `notebooks/12_dlinear_recovery.ipynb` trên Colab (GPU T4), sửa `DRIVE_TRAIN`, `OUT_DIR` ở mục 0 nếu cần, chạy lần lượt các ô.

## Đầu vào
| Mục | Giá trị |
|---|---|
| File train | `train.parquet` Task 6 — SHA-256 `{DATA_SHA256}` (khớp Task 6: {DATA_SHA256 == TASK6_TRAIN_SHA256}) |
| Quy mô | 4.500.000 dòng × 19 cột, 50.000 cặp × 90 ngày → 150.000 mẫu × 480 bước × 6 kênh |
| Eval | Không dùng |

## Mã và môi trường
- Repo nhóm: nhánh `feature/task12-dlinear-recovery`. Mã tác giả `Dingdong-Inc/frn-50k-baseline` @ `{head}`, chạy `latent_demand_recovery/exp/app.py` với `pypots` đi kèm repo; **không sửa file nào của tác giả** (các thay thế nằm trong runner `run_task12.py` ghi ở mục 3 của notebook).
- Python {ENV['python']}, torch {ENV['torch']}, NumPy {ENV['numpy']}, pandas {ENV['pandas']}, pyarrow {ENV['pyarrow']}, pygrinder {ENV['pygrinder']}, tsdb {ENV['tsdb']}; GPU: {gpu}; RAM {ENV['ram_gb']} GB.

## Siêu tham số DLinear (theo `models.py`, giữ nguyên)
`n_steps=480`, `n_features=6`, `moving_avg_window_size=17` (`patch_len // 2 * 2 + 1`, `patch_len=16`), `individual=False`, `d_model=64`, `ORT_weight=MIT_weight=1`, `epochs={EPOCHS}`, `batch_size={BATCH_SIZE}`, `patience=5`, Adam `lr=0.001`, `weight_decay=1e-5`.
> Bảng phân công ghi `individual=True`; mã gốc tác giả truyền `individual=False`. Nhóm giữ đúng mã gốc.

## Thay đổi so với mã tác giả (chỉ trong runner)
- `load_data`, `evaluation_mnar`: giống Task 9 (đọc `train.parquet` local, sắp xếp `store_id, product_id, dt`, X float32 một lần cấp phát, `norm_val` toàn số 1 dạng `(n, 1, 6)`). DLinear dùng chung `DatasetForSAITS` và nhân `imputed_data * inputs['norm_val']` như TimesNet nên kết quả không đổi.
- `_demand_recovery`: đọc `train.parquet` local; chặn âm (đã có trong `_imputation`); chỉ thay `hours_sale_origin[..., 6:22]` tại đúng các giờ hết hàng, giữ nguyên 8 giờ đêm và giờ quan sát được; cộng 24 giờ thành `sale_amount_pred`; ghi Parquet bằng PyArrow (19 cột gốc nguyên vẹn + `sale_amount_pred`, không ghi cột index). So với bản gốc (ghi đè cả lát 6..21 bằng X float32), `sale_amount_pred` chỉ lệch do làm tròn float32: lượt chạy thử đo được `max_abs_diff = {smoke_rec['recovery_max_abs_diff']}`.
- Kiểm chứng tương đương ở lượt chạy thử ({SMOKE_PAIRS} cặp): MNAR `equivalence_ok={smoke_mnar['equivalence_ok']}`; khôi phục `equivalence_ok={smoke_rec['equivalence_ok']}`, `recovery_equivalence_ok={smoke_rec['recovery_equivalence_ok']}`.
- `evaluation_decoupling`, `mimic_missing`, `set_seed`, `_imputation`, mô hình: nguyên mã tác giả.

## Lượt 1 — MNAR (`--missing_rate {MNAR_RATE}`, seed {MNAR_SEED})
| Mục | Nhóm | Bài báo (Bảng 2) |
|---|---|---|
| WAPE | {pct(mnar['wape'])} | {PAPER_MNAR['wape_pct']:.2f}% |
| WPE | {pct(mnar['wpe'])} | {PAPER_MNAR['wpe_pct']:.2f}% |
| Điểm giờ che thêm | {mnar['artificially_masked']:,} (Task 9: {TASK9_MASKED_HOURS:,}) | – |
| Ngày được đánh giá | {mnar['eval_days']:,} (Task 9: {TASK9_EVAL_DAYS:,}) | – |
| Tỷ lệ che thực tế | {mnar['actual_mask_ratio']} | – |

`missing_rate = 0.3` lấy theo ví dụ README của repo tác giả (giống Task 9), chưa xác minh là cấu hình chính xác của bài báo. {'Hoàn tất từ checkpoint `' + MNAR_RESUMED.name + '`.' if MNAR_RESUMED else 'Chạy liền một mạch.'} Thời gian: {mnar['elapsed_sec']} giây; RAM tiến trình cao nhất: {mnar['peak_ram_gb']} GB.

## Lượt 2 — Khôi phục thực tế (`--missing_rate 0`, seed {RECOVERY_SEED})
- Seed thực dùng: `{RECOVERY_SEED}` ({'theo kế hoạch Task 2, giống Task 10' if RECOVERY_SEED == 2024 else 'seed CLI gốc của tác giả'}).
- Số giờ hết hàng được thay: {rec['replaced_stockout_hours']:,}. {'Hoàn tất từ checkpoint `' + REC_RESUMED.name + '`.' if REC_RESUMED else 'Chạy liền một mạch.'} Thời gian: {rec['elapsed_sec']} giây; RAM cao nhất: {rec['peak_ram_gb']} GB.

| Decoupling Score | Nhóm | Bài báo (Bảng 2) |
|---|---|---|
| Raw Sales (`sale_amount`) | {rec['decoupling_raw']:+.6f} | {PAPER_DS['Raw Sales']:+.2f} |
| DLinear (`sale_amount_pred`) | {rec['decoupling_pred']:+.6f} | {PAPER_DS['DLinear']:+.2f} |

## Tệp bàn giao `demand_dlinear.parquet` (cho Task 13, Task 14)
- Quy mô: 4.500.000 dòng × 20 cột (19 cột gốc + `sale_amount_pred`); đối soát `results/demand_file_check.csv`: mọi bộ đếm lỗi = 0; `sale_amount`, `hours_sale` gốc giữ nguyên 100%.
- Kích thước: `{DEMAND_BYTES:,}` bytes | SHA-256: `{DEMAND_SHA256}`
- Đường dẫn Drive: `{DEMAND_DRIVE}`
- Link chia sẻ: {DEMAND_DRIVE_LINK}
- Checkpoint `.pypots` (MNAR và khôi phục): {CKPT_DRIVE_LINK}

## Lệnh đã chạy
{F}
{' '.join(mnar_cmd)}
{' '.join(rec_cmd)}
{F}
'''
(OUT_DIR / 'run_info.md').write_text(info, encoding='utf-8')
print(info)
print('\nFile trong', OUT_DIR)
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file() and 'checkpoints' not in p.parts:
        print(' -', p.relative_to(OUT_DIR), f'({p.stat().st_size:,} bytes)')

# Thông tin chạy Task 12 — DLinear: khôi phục nhu cầu tiềm ẩn

**Cách tái lập:** mở `notebooks/12_dlinear_recovery.ipynb` trên Colab (GPU T4), sửa `DRIVE_TRAIN`, `OUT_DIR` ở mục 0 nếu cần, chạy lần lượt các ô.

## Đầu vào
| Mục | Giá trị |
|---|---|
| File train | `train.parquet` Task 6 — SHA-256 `b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96` (khớp Task 6: True) |
| Quy mô | 4.500.000 dòng × 19 cột, 50.000 cặp × 90 ngày → 150.000 mẫu × 480 bước × 6 kênh |
| Eval | Không dùng |

## Mã và môi trường
- Repo nhóm: nhánh `feature/task12-dlinear-recovery`. Mã tác giả `Dingdong-Inc/frn-50k-baseline` @ `8af694a09a5a427d1d50648d348c022565a781f2`, chạy `latent_demand_recovery/exp/app.py` với `pypots` đi kèm repo; **không sửa file nào của tác giả** (các thay thế nằm trong runner `run_task12.py` ghi ở mục 3 của notebook).
- Python 3.13.16, torch 2.14.1+cu130, NumPy 2.4.6, pandas 2.2.3, pyarrow 23.0.1, pygrinder 0.4, tsdb 0.3.1; GPU: Tesla T4, 15360 MiB; RAM 12.7 GB.

## Siêu th

## Nhận xét (điền sau khi chạy xong)
- **MNAR:** DLinear đạt WAPE …%, WPE …% trên 783.078 ngày; so với Bảng 2 (29,99% / −1,28%) chênh … / … điểm phần trăm. So với TimesNet (Task 9: 27,81% / 2,04%): …
- **Decoupling Score:** Raw Sales … (bài báo −0,57), DLinear … (bài báo −0,16), TimesNet (Task 10) +0,07. Ý nghĩa: …
- **Bàn giao:** `demand_dlinear.parquet` (… bytes, SHA-256 …) đã gửi link cho Task 13 và Task 14 lúc …